In [1]:
# producer_demo.py
import json
import time
from kafka import KafkaProducer

class MessageProducer:
    def __init__(self, bootstrap_servers='localhost:9092'):
        self.producer = KafkaProducer(
            bootstrap_servers=bootstrap_servers,
            # 消息值序列化为 JSON 字节
            value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8'),
            # key 也序列化，用于分区路由
            key_serializer=lambda k: k.encode('utf-8') if k else None,
            acks='all',          # 等待所有副本确认，保证不丢
            retries=3,
        )

    def send(self, topic, key, value):
        """发送一条消息，返回 future 可等待结果"""
        future = self.producer.send(topic, key=key, value=value)
        # 阻塞等待发送结果（生产环境可异步处理）
        record_metadata = future.get(timeout=10)
        print(f"✅ 已发送 -> topic={record_metadata.topic}, "
              f"partition={record_metadata.partition}, offset={record_metadata.offset}")
        return record_metadata

    def close(self):
        self.producer.flush()
        self.producer.close()


if __name__ == "__main__":
    producer = MessageProducer()

    # 模拟发送对话消息
    for i in range(3):
        msg = {
            "session_id": "user_001",
            "role": "user" if i % 2 == 0 else "assistant",
            "content": f"这是第 {i+1} 条测试消息",
            "timestamp": time.time(),
        }
        # key 用 session_id，保证同一会话落到同一分区（有序）
        producer.send("chat-topic", key=msg["session_id"], value=msg)

    producer.close()

/tmp/ipykernel_830/994266870.py:33: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  producer = MessageProducer()
/tmp/ipykernel_830/994266870.py:33: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = MessageProducer()


✅ 已发送 -> topic=chat-topic, partition=0, offset=0
✅ 已发送 -> topic=chat-topic, partition=0, offset=1
✅ 已发送 -> topic=chat-topic, partition=0, offset=2


In [ ]:
# consumer_demo.py
import json
from kafka import KafkaConsumer

class MessageConsumer:
    def __init__(self, topic, group_id='chat-group',
                 bootstrap_servers='localhost:9092'):
        self.consumer = KafkaConsumer(
            topic,
            bootstrap_servers=bootstrap_servers,
            group_id=group_id,                 # 消费组，同组内分摊分区
            auto_offset_reset='earliest',      # 无 offset 时从头读
            enable_auto_commit=True,           # 自动提交 offset（简单场景够用）
            value_deserializer=lambda v: json.loads(v.decode('utf-8')),
            key_deserializer=lambda k: k.decode('utf-8') if k else None,
        )

    def listen(self):
        """持续监听消息"""
        print("👂 开始监听消息...")
        try:
            for record in self.consumer:
                print(f"📥 收到 -> topic={record.topic}, "
                      f"partition={record.partition}, offset={record.offset}, "
                      f"key={record.key}, value={record.value}")
                self.handle(record.value)
        except KeyboardInterrupt:
            print("\n停止消费")
        finally:
            self.consumer.close()

    def handle(self, msg):
        """业务处理：这里可以写落库、调模型、存 Redis 等"""
        print(f"   处理中：session={msg.get('session_id')}, "
              f"role={msg.get('role')}, content={msg.get('content')}")


if __name__ == "__main__":
    consumer = MessageConsumer("chat-topic")
    consumer.listen()

/tmp/ipykernel_830/2266545036.py:8: DeprecationWarning: key_deserializer does not implement kafka.serializer.Deserializer
  self.consumer = KafkaConsumer(
/tmp/ipykernel_830/2266545036.py:8: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  self.consumer = KafkaConsumer(


👂 开始监听消息...
📥 收到 -> topic=chat-topic, partition=0, offset=0, key=user_001, value={'session_id': 'user_001', 'role': 'user', 'content': '这是第 1 条测试消息', 'timestamp': 1794491104.0930593}
   处理中：session=user_001, role=user, content=这是第 1 条测试消息
📥 收到 -> topic=chat-topic, partition=0, offset=1, key=user_001, value={'session_id': 'user_001', 'role': 'assistant', 'content': '这是第 2 条测试消息', 'timestamp': 1794491104.39004}
   处理中：session=user_001, role=assistant, content=这是第 2 条测试消息
📥 收到 -> topic=chat-topic, partition=0, offset=2, key=user_001, value={'session_id': 'user_001', 'role': 'user', 'content': '这是第 3 条测试消息', 'timestamp': 1794491104.3950083}
   处理中：session=user_001, role=user, content=这是第 3 条测试消息
